# Customer Shopping Behaviour — Data Wrangling Capstone
**Author:** Harsh Gupta

**Dataset:** `customer_shopping_behavior.csv`

**Objective:** Load, explore, clean, engineer features, and export the dataset to PostgreSQL.

---
## 1. Import Libraries

In [ ]:
import pandas as pd

---
## 2. Load Dataset

In [ ]:
df = pd.read_csv('customer_shopping_behavior.csv')

---
## 3. Initial Exploration

### 3.1 Head and Tail

In [ ]:
df.head()

In [ ]:
df.tail()

### 3.2 Column Names

In [ ]:
# Fixed: .tolist() must be called as a function, not referenced as a property
print(df.columns.tolist())

### 3.3 Data Types and Summary Statistics

In [ ]:
print(df.dtypes)

In [ ]:
df.info()

In [ ]:
df.describe()

---
## 4. Missing Value Handling

### 4.1 Check for Missing Values

In [ ]:
df.isnull().sum()

### 4.2 Impute Missing Values

`Review Rating` has 37 nulls. Instead of dropping rows, we impute each missing value with the **per-category median** rating — a safe strategy that preserves row count and respects category-level distributions.

In [ ]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
# Confirm no nulls remain
df.isnull().sum()

---
## 5. Duplicate Check

In [ ]:
duplicate_rows = df[df.duplicated()]
print(f"Number of duplicate rows: {len(duplicate_rows)}")

---
## 6. Column Standardisation

### 6.1 Convert Column Names to snake_case

In [ ]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Fixed: .tolist() called correctly as a function
print(df.columns.tolist())

### 6.2 Rename Columns with Special Characters

In [ ]:
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

# Fixed: .tolist() called correctly as a function
print(df.columns.tolist())

---
## 7. Feature Engineering

### 7.1 Age Group

Bin customers into four equal-sized quartile groups using `pd.qcut`.

In [ ]:
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)

In [ ]:
df[['age', 'age_group']].head(10)

### 7.2 Purchase Frequency in Days

Map categorical purchase frequency labels to numeric day intervals.
> Note: `'Bi-Weekly'` and `'Fortnightly'` both map to 14 days as they are equivalent terms. `'Quarterly'` and `'Every 3 Months'` both map to 90 days.

In [ ]:
frequency_mapping = {
    'Weekly': 7,
    'Fortnightly': 14,
    'Bi-Weekly': 14,       # Equivalent to Fortnightly
    'Monthly': 30,
    'Quarterly': 90,
    'Every 3 Months': 90,  # Equivalent to Quarterly
    'Annually': 365
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df[['frequency_of_purchases', 'purchase_frequency_days']].head(10)

---
## 8. Redundant Column Removal

Check whether `discount_applied` and `promo_code_used` carry identical information.

In [ ]:
df[['discount_applied', 'promo_code_used']].head(10)

In [ ]:
# Check if the two columns are identical across all rows
all_equal = (df['discount_applied'] == df['promo_code_used']).all()
print(f"Columns are identical: {all_equal}")

In [ ]:
# Drop the redundant column
df = df.drop('promo_code_used', axis=1)

---
## 9. Final Dataset Preview

In [ ]:
# Fixed: .tolist() called correctly as a function
print(df.columns.tolist())

In [ ]:
df.head(7)

---
## 10. Export to PostgreSQL

Load the cleaned DataFrame into a local PostgreSQL database using SQLAlchemy.

> **Note:** Update the `username`, `password`, and `database` values to match your local PostgreSQL configuration. Avoid committing credentials to version control.

In [ ]:
# Uncomment to install dependencies if not already present
# !pip install psycopg2-binary sqlalchemy

In [ ]:
from sqlalchemy import create_engine

username = "postgres"
password = "root"        # Update before use
host = "localhost"
port = "5432"
database = "customer_behavior"

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

table_name = "customer"

with engine.begin() as conn:
    df.to_sql(
        table_name,
        conn,
        if_exists="replace",  # 'replace' drops and recreates; use 'append' to add rows
        index=False
    )

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")